"""

DATA ENGINEERING ETL PIPELINE - XETRA DATASET

2: Reading multiple files, cleansing data.

Aim:
Write a production ready ETL pipeline using python and pandas.

Overview:
Xetra is a German stock exchange based in Frankfurt operated by Deutsche B?rse Group. 
Data related to daily trading activity is stored publicly on the Amazon S3 database. 
(Update - as of July 2022 the data is no longer available. An archival S3 database will be used) 

Task:
Use jupyter notebook as a protoyping tool to extract and transform source data.
Request and extract source data from cloud based web services.
Use loops and iteration to read and consolidate multiple source files.
Familiarise with pandas package functions to clean and transform output data. 

Below outlines the steps to be performed:
    
    1)Continuation of xetra_1 - requested trading data from Amazon S3 bucket 'xetra-1234'.
    2) Filter multiple xetra buckets by date concatenate data elements into bucket list.
    3) Extract body of data (csv) and read into into pandas dataframe:
        - Initialisation step, where csv column template is taken from first element of bucket list.
        - Iteration step, where remaining list elements are read and appended to pandas dataframe. 
    4) Cleanse data by dropping unecessary rows or columns.  
    5) Print transfomed dataframe object.
    
"""

Data Extraction

In [ ]:
import boto3 #AWS service management package.
import pandas as pd #Data analysis library.
from io import StringIO #String buffer to read CSV files.

In [ ]:
s3 = boto3.resource('s3') #Use the Amazon S3 cloud storage resource.
bucket = s3.Bucket('xetra-1234') #Create instance of the "xetra" data bucket.

In [ ]:
bucket_obj1 = bucket.objects.filter(Prefix='2022-01-28/') #Filter by date and store data as "bucket_obj1".
bucket_obj2 = bucket.objects.filter(Prefix='2022-02-28/') #Filter by date and store data as "bucket_obj2".
bucket_objects = [obj for obj in bucket_obj1] + [obj for obj in bucket_obj2]  #Store data into bucket list.

In [ ]:
#Read csv body of dataset into pandas dataframe - initialisation step:
csv_obj_init = bucket.Object(key=bucket_objects[0].key).get().get('Body') #Initialise first element of csv object.
csv_obj_init = csv_obj_init.read().decode('utf-8') #Store into csv object in utf-8 format.
data = StringIO(csv_obj_init) #Convert csv object from streaming body to string data.
df_init = pd.read_csv(data, delimiter=',') #Read data into pandas data frame.
df_all = pd.DataFrame(columns=df_init.columns) #Initialise df_all with df_init columns.

In [ ]:
#Read csv body of dataset into pandas dataframe - iteration step:
for obj in bucket_objects:
    csv_obj = bucket.Object(key=obj.key).get().get('Body') #Read data element from list.
    csv_obj = csv_obj.read().decode('utf-8') #Store into to csv object in utf-8 format.
    data = StringIO(csv_obj) #Convert csv object to string data.
    df = pd.read_csv(data, delimiter=',') #Read data as pandas data frame.
    df_all = pd.concat([df, df_all]) #Concatenate data to one master dataframe.

In [ ]:
csv_obj #Print csv object to view columns.

Data Cleansing

In [ ]:
#Remove unecessary columns by storing required columns in variable and passing as .loc function argument. 
columns_use = ['ISIN', 'Date', 'Time', 'StartPrice', 'MaxPrice', 'MinPrice', 'EndPrice', 'TradedVolume']
df_all = df_all.loc[:,columns_use]

In [ ]:
df_all.dropna(inplace=True) #Drop all missing values from the dataset.
df_all = df_all.reset_index(drop=True) #Reset the column index.
df_all.shape #Check if there was any filtering (should match table dimensions)

In [ ]:
#Print transformed data output.
df_all